In [1]:
import torch
import torch.nn as nn
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.ops import FeaturePyramidNetwork

class PDC(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        #You can grid_search the best dilation size for your project
        self.d1 = nn.Conv2d(in_channels, out_channels, 3, padding=1, dilation=1)
        self.d2 = nn.Conv2d(in_channels, out_channels, 3, padding=2, dilation=2)
        self.d3 = nn.Conv2d(in_channels, out_channels, 3, padding=3, dilation=3)

        self.fuse = nn.Conv2d(out_channels * 3, out_channels, kernel_size=1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        f1 = self.d1(x)
        f2 = self.d2(x)
        f3 = self.d3(x)

        out = torch.cat([f1, f2, f3], dim=1)
        out = self.fuse(out)
        return self.relu(out)

class PDbackbone(FeaturePyramidNetwork):
    def __init__(self, in_channels_list, out_channels):
        super().__init__(
            in_channels_list=in_channels_list,
            out_channels=out_channels,
            extra_blocks=None
        )

        self.inner_blocks = nn.ModuleList([
            nn.Conv2d(in_channels, out_channels, kernel_size=1)
            for in_channels in in_channels_list
        ])

        self.layer_blocks = nn.ModuleList([
            PDC(out_channels, out_channels)
            for _ in in_channels_list
        ])

#Developing customised Faster_RCNN model with Parallel dilated convolution in backbone
def get_model(num_classes):
    
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")

    in_channels_list = [256, 512, 1024, 2048]
    out_channels = 256

    model.backbone.fpn = PDbackbone(
        in_channels_list=in_channels_list,
        out_channels=out_channels
    )

    model.rpn.anchor_generator = AnchorGenerator(
        sizes=((32,), (64,), (128,), (256,)),
        aspect_ratios=((0.5, 1.0, 2.0),) * 4
    )

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    return model

num_classes = 3 #provide the number of classes for your dataset
model = get_model(num_classes)
print(model)

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(